# <img src="logo.png" width="200" alt="nanoLLaDA">: an end-to-end walkthrough

This tutorial walks through every component of **nanoLLaDA** — a minimal
implementation of the **LLaDA** (Large Language Diffusion with mAsking) paper.

Unlike autoregressive LLMs (GPT-style) that generate tokens left-to-right,
LLaDA is a **masked diffusion language model**: it starts from a fully masked
sequence and iteratively unmasks tokens over multiple steps, similar to how
image diffusion models denoise pixels.

## What you'll learn
0. Setup — download data and train a tokenizer from scratch
1. How the BPE tokenizer works (including the special `<|mask|>` token)
2. The model architecture — a bidirectional Transformer (not causal!)
3. The one-line difference from GPT: `is_causal=False`
4. The forward (noising) process — how we randomly mask tokens for training
5. The diffusion training loss (ELBO bound) and the 1/t weighting
6. The 1% random-length truncation trick
7. Training a small model end-to-end (single GPU)
8. The reverse (denoising) process — iterative unmasking for generation
9. Semi-autoregressive generation with block-wise unmasking
10. Classifier-free guidance (CFG)
11. Multi-GPU training with DDP, gradient accumulation, and `no_sync`
12. `torch.compile` and the `dynamic=True` lesson
13. Checkpointing: saving, auto-cleanup, and graceful resume
14. Loading a trained checkpoint and running inference

This notebook is fully self-contained — it downloads data, trains a tokenizer,
trains a small model, and generates text, all from scratch.

---
## 0. Setup — Download Data & Train Tokenizer

Before we can do anything, we need data and a tokenizer. This cell downloads
a few shards of the ClimbMix dataset and trains a BPE tokenizer on them.
This only needs to run once — subsequent runs will skip if files already exist.

In [ ]:
import os
import subprocess
import sys

# Download 8 data shards (~800MB) if not already present
from nanollada.common import get_base_dir
data_dir = os.path.join(get_base_dir(), "base_data_climbmix")
if not os.path.exists(data_dir) or len([f for f in os.listdir(data_dir) if f.endswith('.parquet')]) < 2:
    print("Downloading dataset shards...")
    subprocess.run([sys.executable, "-m", "nanollada.dataset", "-n", "8"], check=True)
else:
    print(f"Dataset already exists: {len(os.listdir(data_dir))} files in {data_dir}")

# Train tokenizer if not already trained
tokenizer_dir = os.path.join(get_base_dir(), "tokenizer")
if not os.path.exists(os.path.join(tokenizer_dir, "tokenizer.pkl")):
    print("Training tokenizer...")
    subprocess.run([sys.executable, "-m", "scripts.tok_train"], check=True)
else:
    print(f"Tokenizer already trained: {tokenizer_dir}")

---
## 1. Tokenizer

nanoLLaDA uses a custom BPE tokenizer built with `rustbpe` (for training)
and `tiktoken` (for fast encoding at runtime). It has two special tokens:

| Token | Purpose |
|---|---|
| `<\|bos\|>` | Beginning-of-sequence, prepended to every document |
| `<\|mask\|>` | The mask token used during diffusion training and generation |

The mask token is the key difference from autoregressive models — it's what
the model learns to "fill in" during training.

In [ ]:
from nanollada.tokenizer import get_tokenizer

tokenizer = get_tokenizer()

print(f"Vocab size:    {tokenizer.get_vocab_size()}")
print(f"BOS token id:  {tokenizer.get_bos_token_id()}")
print(f"MASK token id: {tokenizer.get_mask_token_id()}")

# Encoding text — returns a list of integer token ids
text = "The capital of France is Paris."
token_ids = tokenizer.encode(text)
print(f"\nText:      {text}")
print(f"Token ids: {token_ids}")
print(f"Decoded:   {tokenizer.decode(token_ids)}")

# Prepend BOS automatically (standard for training/inference)
token_ids_with_bos = tokenizer.encode(text, prepend=tokenizer.get_bos_token_id())
print(f"\nWith BOS:  {token_ids_with_bos}")

# Batch encoding (uses multithreading for speed)
batch = ["Hello world!", "Masked diffusion is cool."]
batch_ids = tokenizer.encode(batch, prepend=tokenizer.get_bos_token_id(), num_threads=2)
for i, ids in enumerate(batch_ids):
    print(f"  [{i}] {ids[:10]}...")

---
## 2. Model Architecture

The `DiffusionTransformer` is a standard Transformer with one critical
difference: **bidirectional attention**. Every token attends to every other
token — including tokens to the right. This is necessary because during
diffusion, the model needs full context to predict masked positions.

Architecture:
- **Embedding** (`wte`): token id → embedding vector
- **RMSNorm** after embedding (no learnable params)
- **N Transformer blocks**, each containing:
  - RMSNorm → Bidirectional Self-Attention (with RoPE + QK-norm) → residual
  - RMSNorm → MLP (ReLU² activation) → residual
- **Final RMSNorm** → **LM head** (projects to vocabulary logits)

Note: the MLP uses **ReLU²** (`F.relu(x).square()`), not SwiGLU or GELU.
ReLU² creates sparser activations and works well for small models.

In [ ]:
from nanollada.model import DiffusionTransformer, DiffusionTransformerConfig
import torch

config = DiffusionTransformerConfig(
    sequence_len=1024,
    vocab_size=32768,
    n_layer=12,
    n_head=12,
    n_embd=768,
)
print(f"Config: {config}")

model = DiffusionTransformer(config)
model.init_weights()
num_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {num_params:,}")

# Forward pass: (B, T) token ids → (B, T, vocab_size) logits
# The input can contain mask tokens — the model predicts what they should be
dummy_input = torch.randint(0, config.vocab_size, (1, 64))
logits = model(dummy_input)
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {logits.shape}")

---
## 3. The One-Line Difference from GPT

The entire architectural difference between an autoregressive model (GPT)
and a masked diffusion model (LLaDA) is **one boolean**:

```python
# GPT (autoregressive) — token i can only see tokens 0..i
y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

# LLaDA (masked diffusion) — every token sees every other token
y = F.scaled_dot_product_attention(q, k, v, is_causal=False)
```

Everything else — RoPE, RMSNorm, ReLU² MLP, the transformer structure — is
identical. This is why LLaDA can reuse the same architecture as GPT/LLaMA.

**Why bidirectional?** When predicting a masked token, the model benefits from
seeing tokens on BOTH sides. In GPT, predicting token 5 only uses tokens 0–4.
In LLaDA, predicting a masked token at position 5 uses ALL unmasked tokens,
including those at positions 6, 7, 8... This is strictly more information.

The tradeoff is generation speed: GPT generates one token per forward pass
(with KV cache), while LLaDA needs N forward passes to iteratively unmask.

In [ ]:
import torch.nn.functional as F

# Demonstrate the difference
q = torch.randn(1, 4, 8, 32)  # (B, H, T, D)
k = torch.randn(1, 4, 8, 32)
v = torch.randn(1, 4, 8, 32)

# Causal: each position only attends to itself and earlier positions
causal_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

# Bidirectional: each position attends to ALL positions
bidir_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

# They differ because bidirectional uses more context
print(f"Causal output[0,0,4,:4]:       {causal_out[0,0,4,:4].tolist()}")
print(f"Bidirectional output[0,0,4,:4]: {bidir_out[0,0,4,:4].tolist()}")

---
## 4. The Forward Process (Noising)

During training, we corrupt clean sequences by randomly replacing tokens with
`<|mask|>`. The mask ratio `t` is sampled uniformly from `[ε, 1]` for each
sequence in the batch.

Key details:
- At `t≈0` the sequence is nearly clean, at `t=1` it's fully masked
- Each sequence in the batch gets a DIFFERENT random mask ratio
- **Position 0 (BOS) is never masked** — it anchors the sequence and provides
  a fixed reference point. Without it, a fully-masked sequence would have zero
  information for the model to work with. During generation, BOS is always in
  the prompt (never masked), so training should match this.

In [ ]:
mask_id = tokenizer.get_mask_token_id()

from nanollada.diffusion import forward_process

# Demo
clean = torch.tensor([[tokenizer.get_bos_token_id()] + tokenizer.encode("Diffusion models are amazing")])
noisy, mask, p = forward_process(clean, mask_id)

print("Clean tokens: ", clean[0].tolist())
print("Noisy tokens: ", noisy[0].tolist())
print("Mask ratio:   ", f"{p[0, 0].item():.2f}")
print(f"Masked {mask.sum().item()}/{clean.shape[1]} tokens")
print(f"Position 0 masked? {mask[0, 0].item()} (should be False)")


---
## 5. Training Loss and the 1/t Weighting

The loss is **cross-entropy on masked positions only**, weighted by
`1 / mask_ratio`. This weighting is the mathematical core of LLaDA.

**Why divide by mask_ratio?**
- When mask ratio is high (90%): many masked tokens, but each prediction has
  little context → each individual prediction is "cheap"
- When mask ratio is low (10%): few masked tokens, but each prediction has
  lots of context → each individual prediction is "expensive"

Without the 1/t weighting, high mask ratios would dominate the loss (more
terms). The weighting balances all mask ratios equally, and mathematically
makes the loss an **ELBO** (evidence lower bound) on the negative
log-likelihood. This is what makes LLaDA a proper generative model, not BERT.


In [ ]:
from nanollada.diffusion import compute_diffusion_loss

dummy_batch = torch.randint(0, config.vocab_size, (2, 64))
loss = compute_diffusion_loss(model, dummy_batch, mask_id)
print(f"Diffusion loss: {loss.item():.4f}")

---
## 6. The 1% Random-Length Trick

From the LLaDA paper: 1% of the time, we randomly truncate the input
sequence to a random length before masking:

```python
if torch.rand(1).item() < 0.01:
    random_length = torch.randint(1, input_ids.shape[1] + 1, (1,)).item()
    input_ids = input_ids[:, :random_length]
```

This helps the model handle variable-length inputs at inference time.
Without it, the model only ever sees full-length (e.g. 1024 token) sequences
and may behave oddly on shorter inputs.

**Practical gotcha**: when using `torch.compile(dynamic=False)`, this causes
a recompilation for the new sequence length, which allocates a second compiled
graph in GPU memory. On tight-memory setups (like our 4×L4 with a d20 model),
this caused an OOM crash. The fix: use `torch.compile(dynamic=True)` so the
compiler generates shape-generic code that handles any sequence length without
recompiling.

---
## 7. Train a Small Model End-to-End

Let's actually train a tiny model right here in the notebook! We'll use a
depth=4 model (20M params) with a short sequence length, train for 100 steps,
and then generate text from it. This takes ~1 minute on a single GPU.

In [ ]:
from nanollada.dataloader import distributed_data_loader

# Build a tiny model
train_config = DiffusionTransformerConfig(
    sequence_len=512, vocab_size=tokenizer.get_vocab_size(),
    n_layer=4, n_head=4, n_embd=256,
)
train_model = DiffusionTransformer(train_config)
train_model.init_weights()
device = "cuda" if torch.cuda.is_available() else "cpu"
train_model = train_model.to(device)
print(f"Training model: {sum(p.numel() for p in train_model.parameters()):,} params on {device}")

# Setup optimizer and dataloader
optimizer = torch.optim.AdamW(train_model.parameters(), lr=3e-4, betas=(0.9, 0.95))
loader = distributed_data_loader(tokenizer, B=4, T=512, split="train", device=device)

# Train for 100 steps
train_model.train()
for step in range(100):
    input_ids, _ = next(loader)
    noisy, masked, p = forward_process(input_ids, mask_id)
    logits = train_model(noisy)
    token_loss = F.cross_entropy(logits[masked], input_ids[masked], reduction='none') / p[masked]
    loss = token_loss.sum() / (input_ids.shape[0] * input_ids.shape[1])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    if step % 20 == 0:
        print(f"  step {step:3d} | loss: {loss.item():.4f}")

print("Training complete!")

# Generate from our freshly trained model
from nanollada.generate import generate

train_model.eval()
prompt_text = "The capital of France is"
tokens = tokenizer.encode(prompt_text, prepend=tokenizer.get_bos_token_id())
prompt_tensor = torch.tensor([tokens], dtype=torch.long, device=device)
with torch.no_grad():
    out = generate(train_model, prompt_tensor, mask_id, steps=32, gen_length=32, temperature=0.)
print(f"\nPrompt: {prompt_text}")
print(f"Output: {tokenizer.decode(out[0, len(tokens):].tolist())}")
print("(Gibberish is expected — 100 steps on a tiny model is just a proof of concept! My run gives me 'the the the ...')")
print("What is more important here is that you should see loss going down pretty rapidly - since there is no warmup schedule")

---
## 8. Generation: Iterative Unmasking

Generation is the **reverse** of the forward process. Start with a prompt
followed by all `<|mask|>` tokens, then iteratively unmask:

```
Step 0: [The capital of France is] [MASK] [MASK] [MASK] [MASK]
Step 1: [The capital of France is] [MASK] [MASK] [MASK]  .
Step 2: [The capital of France is] [MASK]  Paris  [MASK]  .
Step 3: [The capital of France is] [MASK]  Paris  ,       .
Step 4: [The capital of France is]  indeed  Paris  ,       .
```

At each step:
1. Run the model on the current (partially masked) sequence
2. Sample a candidate token for each masked position (via Gumbel-max)
3. Score each candidate by confidence (softmax probability of the prediction)
4. Unmask the top-k most confident predictions
5. The rest stay masked for the next iteration

The number of tokens unmasked per step follows a uniform schedule.
Steps are clamped to the mask count — no wasted forward passes if
`steps > gen_length`.

In [ ]:
from nanollada.generate import generate, add_gumbel_noise, get_num_transfer_tokens

# --- Unmasking schedule ---
mask_index = torch.ones(1, 16, dtype=torch.bool)
schedule = get_num_transfer_tokens(mask_index, steps=4)
print("16 masked tokens, 4 steps → tokens per step:", schedule[0].tolist())

# Steps clamped when > mask count
schedule2 = get_num_transfer_tokens(mask_index, steps=100)
print(f"16 masked tokens, 100 steps → clamped to {schedule2.shape[1]} steps:", schedule2[0].tolist())

# --- Gumbel noise ---
logits = torch.tensor([[1.0, 2.0, 0.5]])
print(f"\nGreedy (temp=0): token {torch.argmax(add_gumbel_noise(logits, 0.0), dim=-1).item()}")
samples = [torch.argmax(add_gumbel_noise(logits, 1.0), dim=-1).item() for _ in range(20)]
print(f"Stochastic (temp=1.0), 20 samples: {samples}")

---
## 9. Semi-Autoregressive Generation (Block-wise)

Fully parallel generation unmasks ALL positions at once. This can struggle
with long sequences because distant tokens have weak dependencies.

**Semi-autoregressive** generation splits the output into blocks and
generates them left-to-right:

```
block_length=4, gen_length=8, steps=8 → 2 blocks, 4 steps each

Block 1: [prompt] [M M M M] [M M M M]  ← unmask first 4
Block 2: [prompt] [a b c d] [M M M M]  ← then unmask last 4
```

Each block can see all previous blocks as context, giving a middle ground
between fully parallel (fast) and fully autoregressive (coherent).

Semi-autoregressive configurations:
- gen_length=64, block_length=64 → 1 block  (fully parallel)
- gen_length=64, block_length=32 → 2 blocks
- gen_length=64, block_length=16 → 4 blocks
- gen_length=64, block_length=8  → 8 blocks (most coherent)

---
## 10. Classifier-Free Guidance (CFG)

CFG amplifies the prompt's influence on generation:

```python
guided = unconditional + (1 + scale) * (conditional - unconditional)
```

- `cfg_scale=0` → no guidance (just conditional logits)
- `cfg_scale=1` → prompt influence doubled
- `cfg_scale=2` → prompt influence tripled

The "unconditional" input is created by masking the prompt tokens too.
This is **unsupervised** — no special training needed. The model already
learned to handle masked prompts during pretraining.

Tradeoff: doubles compute per step (two forward passes).

---
## 11. Multi-GPU Training with DDP

nanoLLaDA uses PyTorch's DistributedDataParallel (DDP) for multi-GPU training:

```bash
torchrun --standalone --nproc_per_node=4 -m scripts.train
```

How it works:
- Each GPU (rank) runs the same training script independently
- Each rank reads **different data** (dataloader shards by rank)
- After `.backward()`, DDP **averages gradients** across all ranks via AllReduce
- All ranks take the same optimizer step → models stay in sync

### Gradient Accumulation and `no_sync`

When the total batch size is larger than what fits in one forward pass,
we accumulate gradients over multiple micro-steps:

```python
for micro_step in range(grad_accum_steps):
    is_last = (micro_step == grad_accum_steps - 1)
    ctx = model.no_sync if (ddp and not is_last) else lambda: nullcontext()
    with ctx():
        loss = compute_loss(...)
        (loss / grad_accum_steps).backward()
optimizer.step()
```

**`model.no_sync()`** suppresses the AllReduce on all but the last micro-step.
Without it, DDP would communicate gradients on EVERY backward pass — wasting
bandwidth. We only need the final averaged gradient.

**Important**: the model must be wrapped in DDP BEFORE `torch.compile`:
```python
model = DDP(model, device_ids=[local_rank])  # DDP first
model = torch.compile(model, dynamic=True)    # compile second
```
This ensures `model.no_sync` is available (it's a DDP method, not a
base Module method).

---
## 12. `torch.compile` and `dynamic=True`

We use `torch.compile` to JIT-compile the model for faster training:

```python
model = torch.compile(model, dynamic=True)
```

**Why `dynamic=True`?** The 1% random-length truncation changes the sequence
length. With `dynamic=False`, torch.compile generates code for a fixed shape.
When a different shape appears, it recompiles — allocating a SECOND compiled
graph in GPU memory. On our 4×L4 setup with the d20 model (477M params,
15.6GB steady state), this extra memory pushed us over the 23GB limit and
caused an OOM crash at step 3559.

`dynamic=True` generates shape-generic code using symbolic shapes, so any
sequence length works without recompilation. Slightly slower per step (~2-5%)
but no OOM risk.

---
## 13. Checkpointing

### What's saved
Each checkpoint consists of:
- `model_NNNNNN.pt` — model weights (~1.7GB for d20)
- `meta_NNNNNN.json` — config, step, LR state, dataloader position
- `optim_NNNNNN_rankN.pt` — optimizer state per DDP rank (~1.1GB each)

### Auto-cleanup
Checkpoints are large (model + 4 optimizer shards ≈ 6GB for d20). Saving
every 1000 steps for a 35K-step run would be 210GB of checkpoints. We learned
this the hard way — the disk filled to 100% at step 20000 and crashed the run.

The fix: `save_checkpoint` now auto-deletes old checkpoints, keeping only the
last 3:

```python
def save_checkpoint(..., keep_last=3):
    # ... save new checkpoint ...
    _cleanup_old_checkpoints(checkpoint_dir, keep_last)
```

### Graceful resume
If the optimizer files are corrupt (e.g. disk full during write), the loader
handles it gracefully instead of crashing:

```python
# load_checkpoint returns optimizer_data=None if files are missing/corrupt
# train.py handles this:
if resuming and optimizer_data is not None:
    optimizer.load_state_dict(optimizer_data)
elif resuming:
    print("WARNING: No optimizer state, resuming with fresh optimizer")
```

A fresh optimizer loses the AdamW momentum/variance history, so the first
few hundred steps after resume are slightly less efficient. But the model
weights are intact and training recovers quickly.

---
## 14. Loading a Checkpoint & Running Inference

In [ ]:
import os
import json
from nanollada.checkpoint import load_checkpoint
from nanollada.common import get_base_dir

# Auto-detect the best available checkpoint
base_dir = get_base_dir()
checkpoint_root = os.path.join(base_dir, "checkpoints")
trained_model = None

if os.path.exists(checkpoint_root):
    # Find all model tags, prefer d20-full > d12-full > d12 > anything
    tags = sorted(os.listdir(checkpoint_root), reverse=True)
    for tag in tags:
        tag_dir = os.path.join(checkpoint_root, tag)
        model_files = sorted([f for f in os.listdir(tag_dir) if f.startswith("model_") and f.endswith(".pt")])
        if model_files:
            latest_step = int(model_files[-1].split("_")[1].split(".")[0])
            print(f"Found checkpoint: {tag} at step {latest_step}")

            with open(os.path.join(tag_dir, f"meta_{latest_step:06d}.json")) as f:
                meta = json.load(f)
            model_config = DiffusionTransformerConfig(**meta["model_config"])

            device = "cuda" if torch.cuda.is_available() else "cpu"
            trained_model = DiffusionTransformer(model_config)
            model_data, _, _ = load_checkpoint(tag_dir, latest_step, device, load_optimizer=False)
            trained_model.load_state_dict(model_data, strict=True)
            trained_model.to(device).eval()
            print(f"Loaded: {sum(p.numel() for p in trained_model.parameters()):,} params on {device}")
            print(f"Config: {meta['model_config']}")
            break

if trained_model is None:
    print("No checkpoint found. Run 'bash run.sh' to train a model first.")

---
## 15. Putting It All Together: Generate & Compare

Let's generate text with different strategies and compare their speed.
Each mode trades off quality vs compute differently.

In [ ]:
# Helper to generate and time a single prompt
import time

def timed_generate(model, prompt_text, tokenizer, mask_id, **kwargs):
    tokens = tokenizer.encode(prompt_text, prepend=tokenizer.get_bos_token_id())
    prompt_tensor = torch.tensor([tokens], dtype=torch.long, device=model.get_device())
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = generate(model, prompt_tensor, mask_id, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.time() - t0
    text = tokenizer.decode(out[0, len(tokens):].tolist())
    return text, dt

In [ ]:
# Greedy, fully parallel. All masked tokens are unmasked in one block.
# Fastest mode. 1 block x 64 steps = 64 forward passes

prompt = "The capital of France is"

if trained_model is not None:
    text, dt = timed_generate(trained_model, prompt, tokenizer, mask_id,
                               steps=64, gen_length=64, temperature=0.)
    print(f"Output: {text}")
    print(f"Time:   {dt*1000:.0f}ms")

In [ ]:
# With temperature (stochastic sampling)
# Adds Gumbel noise for diversity. Same speed as greedy.

if trained_model is not None:
    text, dt = timed_generate(trained_model, prompt, tokenizer, mask_id,
                               steps=64, gen_length=64, temperature=0.5)
    print(f"Output: {text}")
    print(f"Time:   {dt*1000:.0f}ms")

In [ ]:
# Semi-autoregressive (block_length=16)
# Generates in 4 blocks of 16 tokens left-to-right. More coherent, same compute.
# 4 blocks x 16 steps = 64 forward passes, same as fully parallel

if trained_model is not None:
    text, dt = timed_generate(trained_model, prompt, tokenizer, mask_id,
                               steps=64, gen_length=64, block_length=16, temperature=0.5)
    print(f"Output: {text}")
    print(f"Time:   {dt*1000:.0f}ms")

In [ ]:
# Classifier-free guidance (cfg_scale=1.0)
# Amplifies prompt relevance. **2x slower** — runs the model twice per step.

if trained_model is not None:
    text, dt = timed_generate(trained_model, prompt, tokenizer, mask_id,
                               steps=64, gen_length=64, block_length=16,
                               temperature=0.5, cfg_scale=1.0)
    print(f"Output: {text}")
    print(f"Time:   {dt*1000:.0f}ms")

In [ ]:
# More steps = higher quality, slower
# Compare 16 vs 64 vs 128 unmasking steps.

if trained_model is not None:
    for steps in [16, 32, 64]:
        text, dt = timed_generate(trained_model, prompt, tokenizer, mask_id,
                                   steps=steps, gen_length=64, temperature=0.5)
        print(f"steps={steps:3d} | {dt*1000:6.0f}ms | {text[:80]}...")

---
## Summary

Key takeaways:
- LLaDA replaces autoregressive generation with masked diffusion
- The architecture difference is one boolean: `is_causal=False`
- Training: randomly mask tokens, predict them, weight loss by `1/mask_ratio`
- The 1/t weighting makes the loss an ELBO — a proper generative model
- Generation: start fully masked, iteratively unmask most-confident tokens
- Semi-autoregressive blocks trade off speed vs coherence
- CFG amplifies prompt-relevance (free at inference, no special training)
- Use `torch.compile(dynamic=True)` when sequence lengths vary
- Auto-cleanup checkpoints to avoid filling the disk
- Handle corrupt optimizer state gracefully on resume
